## Homework Assignment

1. **Find the busiest day of each month using window functions**
   - Group flights by month and day
   - Count flights per day
   - Use ROW_NUMBER() to rank days within each month
   - Return only the busiest day per month

2. **Calculate week-over-week change in average delays**
   - Aggregate average delays by week (use WEEKOFYEAR or DATE_TRUNC)
   - Use LAG to get the previous week's average
   - Calculate both absolute and percentage change

3. **Rank airlines by on-time performance for each month**
   - Define "on-time" as arr_delay <= 0
   - Calculate the on-time percentage for each carrier per month
   - Rank carriers within each month by on-time percentage

4. **Identify flights that were worse than the carrier's monthly average**
   - Use a window function to calculate each carrier's monthly average delay
   - Find flights where arr_delay exceeds the carrier's monthly average
   - Return the top 20 worst offenders relative to their carrier's average

In [0]:
%sql
-- One day for each month with highest/busiest day each month 
SELECT
month,day,
total_flights
FROM (
SELECT
    month, day,
    COUNT(*) AS total_flights,
    ROW_NUMBER() OVER (PARTITION BY month ORDER BY COUNT(*) DESC) AS top_day
FROM flights
    GROUP BY
    month,
    day
)
WHERE top_day = 1
ORDER BY month asc;

month,day,total_flights
1,2,943
2,28,964
3,14,982
4,15,995
5,30,989
6,20,995
7,11,1006
8,7,1001
9,13,996
10,3,995


In [0]:
%sql
--2.Calculate week-over-week change in average delays
--Aggregate average delays by week (use WEEKOFYEAR or DATE_TRUNC)
--Use LAG to get the previous week's average
--Calculate both absolute and percentage change
SELECT
week_num,
prev_week_delay,
avg_delay,
--Absolute and percentage change
ROUND(avg_delay - prev_week_delay, 2) AS abs_change,
ROUND(((avg_delay - prev_week_delay) / prev_week_delay)* 100, 2) AS percent_change
FROM (
  SELECT
  WEEKOFYEAR(MAKE_DATE(year, month, day)) AS week_num,
  ROUND(AVG(arr_delay), 2) AS avg_delay,
 -- Use LAG to get the previous week's average
  LAG(ROUND(AVG(arr_delay), 2)) OVER (ORDER BY WEEKOFYEAR (MAKE_DATE(year, month, day))) AS prev_week_delay
  FROM flights
  WHERE arr_delay IS NOT NULL
  GROUP BY week_num
)
ORDER BY week_num;

week_num,prev_week_delay,avg_delay,abs_change,percent_change
1,null,6.21,null,null
2,6.21,-2.38,-8.59,-138.33
3,-2.38,6.36,8.74,-367.23
4,6.36,10.1,3.74,58.81
5,10.1,9.34,-0.76,-7.52
6,9.34,6.18,-3.16,-33.83
7,6.18,5.48,-0.7,-11.33
8,5.48,6.63,1.15,20.99
9,6.63,2.13,-4.5,-67.87
10,2.13,12.52,10.39,487.79


In [0]:
%sql
-- 3.Rank airlines by on-time performance for each month
--Define "on-time" as arr_delay <= 0
--Calculate the on-time percentage for each carrier per month
--Rank carriers within each month by on-time percentage
SELECT
  month,
  carrier,
  on_time,
  -- Rank carriers within each month by on-time percentage
  DENSE_RANK() OVER (PARTITION BY month ORDER BY on_time DESC) AS ontime_pct_group
FROM (
-- Calculate the on-time percentage for each carrier per month
  SELECT
  month,
  carrier,
-- Define "on-time" as arr_delay <= 0
 ROUND(100.0 * SUM(CASE WHEN arr_delay <= 0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS on_time
    FROM flights
    WHERE arr_delay IS NOT NULL
    GROUP BY
    month,
    carrier)
ORDER BY month, ontime_pct_group;

month,carrier,on_time,ontime_pct_group
1,VX,82.80,1
1,HA,77.42,2
1,DL,70.86,3
1,AA,63.47,4
1,US,61.71,5
1,B6,58.64,6
1,UA,57.91,7
1,9E,57.77,8
1,WN,55.74,9
1,FL,54.63,10


In [0]:
%sql
--4.Identify flights that were worse than the carrier's monthly average
--Use a window function to calculate each carrier's monthly average delay
--Find flights where arr_delay exceeds the carrier's monthly average
--Return the top 20 worst offenders relative to their carrier's average
SELECT
carrier,
month,
flight,
arr_delay,
avg_monthly_delay,
(arr_delay - avg_monthly_delay) AS delay_diff
FROM (
    SELECT
    carrier,
    flight,
    arr_delay,
    month,
    AVG(arr_delay) OVER(PARTITION BY carrier, month) AS avg_monthly_delay
    FROM flights
    WHERE arr_delay IS NOT NULL)
WHERE arr_delay > avg_monthly_delay
ORDER BY delay_diff DESC

carrier,month,flight,arr_delay,avg_monthly_delay,delay_diff
HA,1,51,1272,27.483870967741936,1244.516129032258
MQ,6,3535,1127,23.14372163388805,1103.8562783661118
MQ,1,3695,1109,7.883794825238311,1101.1162051747617
AA,9,177,1007,-8.57398753894081,1015.5739875389409
MQ,7,3075,989,22.744549763033174,966.2554502369668
DL,4,2391,931,3.444362745098039,927.555637254902
DL,3,2119,915,1.4148884578079535,913.5851115421921
DL,7,2047,895,14.920631125986134,880.0793688740139
AA,12,172,878,7.7196765498652296,870.2803234501348
MQ,5,3744,875,11.939007737824307,863.0609922621757
